# LeetCode 683 — K Empty Slots bằng Python

- Link: https://leetcode.com/problems/k-empty-slots/
- Độ khó: **Hard** (bài **Premium**)
- Chủ đề: Mô phỏng, Tìm kiếm nhị phân trên danh sách đã sắp (`bisect`), Đảo mảng (vị trí ↔ ngày), Cửa sổ trượt, Chia xô (bucket)

> **Ghi chú Premium:** Đây là bài dành cho tài khoản Premium. Phần đề bài dưới đây được viết lại theo hiểu biết về bài (nội dung đề, ví dụ và ràng buộc phổ biến). Nếu bạn có tài khoản Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc trên trang LeetCode.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def kEmptySlots(self, bulbs: List[int], k: int) -> int:

```

## 1. Giới thiệu & mục tiêu học

Có một hàng bóng đèn, mỗi ngày bật thêm đúng một bóng. Ta muốn biết **ngày sớm nhất** xuất hiện hai bóng đang sáng mà ở giữa chúng có **đúng `k` bóng**, tất cả đều đang tắt.

Bài này rất hay để luyện một kỹ năng quan trọng: **đổi góc nhìn dữ liệu**. Mảng đề cho là "ngày → vị trí"; nếu đảo lại thành "vị trí → ngày", bài toán mô phỏng theo thời gian biến thành bài **cửa sổ trượt** trên mảng tĩnh. Ta cũng gặp một mẹo đẹp: **chia xô kích thước `k + 1`**.

Sau notebook này bạn sẽ:

1. Mô phỏng đúng bài toán và viết brute force làm "trọng tài".
2. Hiểu nhận xét then chốt: khi bật một bóng, **chỉ cần nhìn hai bóng sáng gần nhất hai bên** nó.
3. Dùng `bisect` để tìm hàng xóm sáng gần nhất trong danh sách đã sắp.
4. Đảo mảng thành `days[pos]` và giải bằng cửa sổ trượt `O(n)`.
5. Hiểu mẹo chia xô `k + 1` (chỉ cần min/max của mỗi xô) và kiểm chứng mọi cách bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Có `n` bóng đèn xếp thành một hàng, đánh số vị trí từ `1` đến `n`, ban đầu tất cả đều tắt. Mỗi ngày ta bật **đúng một** bóng; sau `n` ngày tất cả đều sáng.

Cho mảng `bulbs` độ dài `n`, trong đó `bulbs[i] = x` nghĩa là: vào **ngày `i + 1`**, bóng ở **vị trí `x`** được bật (chỉ số `i` đếm từ 0, ngày đếm từ 1).

Cho số nguyên `k`. Trả về **ngày nhỏ nhất** sao cho tồn tại hai bóng **đang sáng** mà giữa chúng có **đúng `k` bóng** và **cả `k` bóng đó đều đang tắt**. Nếu không có ngày nào như vậy, trả về `-1`.

**Ví dụ 1**

```text
Input:  bulbs = [1, 3, 2], k = 1
Output: 2
Giải thích:
  Ngày 1: bật vị trí 1 → [1, 0, 0]
  Ngày 2: bật vị trí 3 → [1, 0, 1]   ← vị trí 1 và 3 sáng, giữa là 1 bóng tắt ✓
  Ngày 3: bật vị trí 2 → [1, 1, 1]
Ngày sớm nhất là 2.
```

**Ví dụ 2**

```text
Input:  bulbs = [1, 2, 3], k = 1
Output: -1
```

**Ràng buộc (theo hiểu biết, nên đối chiếu):**

- `n == bulbs.length`, `1 <= n <= 2 * 10^4`
- `1 <= bulbs[i] <= n`, `bulbs` là một **hoán vị** của `1..n`
- `0 <= k <= 2 * 10^4`

In [1]:
def show_days(bulbs):
    """In trạng thái hàng bóng sau từng ngày (1 = sáng, . = tắt)."""
    n = len(bulbs)
    state = [0] * (n + 1)          # dùng chỉ số 1..n cho khớp đề
    for day, x in enumerate(bulbs, start=1):
        state[x] = 1
        row = ''.join('1' if state[p] else '.' for p in range(1, n + 1))
        print(f"ngày {day}: bật vị trí {x:>2} -> {row}")

show_days([1, 3, 2])
print()
show_days([1, 2, 3])

ngày 1: bật vị trí  1 -> 1..
ngày 2: bật vị trí  3 -> 1.1
ngày 3: bật vị trí  2 -> 111

ngày 1: bật vị trí  1 -> 1..
ngày 2: bật vị trí  2 -> 11.
ngày 3: bật vị trí  3 -> 111


## 3. Kiến thức nền

### 3.1. `enumerate(xs, start=1)`

`enumerate` cho cả chỉ số lẫn giá trị. Tham số `start=1` để đếm ngày từ 1 cho khớp đề.

In [2]:
for day, pos in enumerate([1, 3, 2], start=1):
    print("ngày", day, "bật vị trí", pos)

ngày 1 bật vị trí 1
ngày 2 bật vị trí 3
ngày 3 bật vị trí 2


### 3.2. Đảo mảng hoán vị: "ngày → vị trí" thành "vị trí → ngày"

Vì `bulbs` là hoán vị, mỗi vị trí được bật đúng một lần. Ta có thể lập `days[p] = ngày bóng p được bật`. Hai mảng chứa **cùng thông tin**, chỉ khác góc nhìn.

In [3]:
bulbs = [3, 1, 5, 4, 2]
n = len(bulbs)
days = [0] * n                      # days[p-1] = ngày bật vị trí p (dùng chỉ số 0)
for day, x in enumerate(bulbs, start=1):
    days[x - 1] = day
print("bulbs (ngày -> vị trí):", bulbs)
print("days  (vị trí -> ngày):", days)

bulbs (ngày -> vị trí): [3, 1, 5, 4, 2]
days  (vị trí -> ngày): [2, 5, 1, 4, 3]


### 3.3. `bisect` — tìm vị trí chèn trong danh sách đã sắp

- `bisect_left(a, x)`: chỉ số đầu tiên `i` mà `a[i] >= x`. Mọi phần tử bên trái `i` đều `< x`.
- Vậy hàng xóm nhỏ hơn gần nhất là `a[i-1]` (nếu `i > 0`), hàng xóm lớn hơn gần nhất là `a[i]` (nếu `i < len(a)`).
- `a.insert(i, x)` chèn mà vẫn giữ thứ tự (tốn `O(len(a))` vì phải dời phần tử).

In [4]:
from bisect import bisect_left, insort

on = [2, 7, 11]
x = 9
i = bisect_left(on, x)
print("i =", i, "| hàng xóm trái:", on[i - 1] if i > 0 else None, "| hàng xóm phải:", on[i] if i < len(on) else None)
insort(on, x)                       # = on.insert(bisect_left(on, x), x)
print("sau insort:", on)

i = 2 | hàng xóm trái: 7 | hàng xóm phải: 11
sau insort: [2, 7, 9, 11]


### 3.4. Phép chia nguyên để chia xô

`p // size` cho biết vị trí `p` thuộc xô thứ mấy khi chia hàng thành các đoạn liên tiếp dài `size`. Với `size = 3`: vị trí 0,1,2 → xô 0; 3,4,5 → xô 1; …

In [5]:
size = 3
print([(p, p // size) for p in range(10)])

[(0, 0), (1, 0), (2, 0), (3, 1), (4, 1), (5, 1), (6, 2), (7, 2), (8, 2), (9, 3)]


### 3.5. "Vô cực" trong Python

Khi tìm giá trị nhỏ nhất, khởi tạo bằng `float('inf')`; mọi số đều nhỏ hơn nó. Cuối cùng nếu vẫn là `inf` nghĩa là "không tìm thấy".

In [6]:
INF = float('inf')
best = INF
for v in [7, 3, 9]:
    best = min(best, v)
print(best, INF > 10**18)

3 True


## 4. Ý tưởng nền tảng / trực giác

### 4.1. "Đúng `k` bóng tắt ở giữa" nghĩa là gì?

Hai bóng sáng ở vị trí `a < b`, giữa chúng có `b - a - 1` bóng. Điều kiện:

- `b - a - 1 == k`, tức là `b - a == k + 1`;
- mọi vị trí `a+1, …, b-1` đều đang tắt — tức là `a` và `b` là **hai bóng sáng kề nhau** (không có bóng sáng nào giữa chúng).

Vậy câu hỏi mỗi ngày là: **trong dãy các vị trí đang sáng (đã sắp xếp), có hai phần tử liên tiếp nào cách nhau đúng `k + 1` không?**

### 4.2. Tính tay

`bulbs = [3, 1, 5, 4, 2]`, `k = 1` (cần hai bóng sáng kề nhau cách đúng 2):

| Ngày | Bật | Các vị trí đang sáng (sắp xếp) | Khoảng cách các cặp liên tiếp | Có cặp = 2? |
|---|---|---|---|---|
| 1 | 3 | `[3]` | — | không |
| 2 | 1 | `[1, 3]` | 2 | **có** → đáp án **2** |

Một ví dụ khác `bulbs = [1, 5, 3, 2, 4]`, `k = 1`:

| Ngày | Bật | Đang sáng | Khoảng cách | Có cặp = 2? |
|---|---|---|---|---|
| 1 | 1 | `[1]` | — | không |
| 2 | 5 | `[1, 5]` | 4 | không |
| 3 | 3 | `[1, 3, 5]` | 2, 2 | **có** → đáp án **3** |

### 4.3. Quan sát quan trọng: chỉ cặp chứa bóng mới bật là "mới"

Bật thêm bóng `x` chỉ **phá** các cặp cũ (nếu `x` nằm giữa chúng) và **tạo** đúng hai cặp kề mới: (hàng xóm sáng bên trái, `x`) và (`x`, hàng xóm sáng bên phải). Vì ta tìm **ngày đầu tiên**, mỗi ngày chỉ cần kiểm tra hai cặp mới này.

In [7]:
def sorted_on_positions(bulbs, day):
    return sorted(bulbs[:day])

for day in range(1, 6):
    on = sorted_on_positions([1, 5, 3, 2, 4], day)
    gaps = [b - a for a, b in zip(on, on[1:])]
    print(f"ngày {day}: sáng {on}, khoảng cách {gaps}")

ngày 1: sáng [1], khoảng cách []
ngày 2: sáng [1, 5], khoảng cách [4]
ngày 3: sáng [1, 3, 5], khoảng cách [2, 2]
ngày 4: sáng [1, 2, 3, 5], khoảng cách [1, 1, 2]
ngày 5: sáng [1, 2, 3, 4, 5], khoảng cách [1, 1, 1, 1]


## 5. Cách 1 — Brute Force: mô phỏng và quét cả hàng mỗi ngày

**Ý tưởng.** Mỗi ngày bật một bóng, rồi quét toàn bộ hàng từ trái sang phải, nhớ vị trí bóng sáng gần nhất vừa gặp (`prev`). Mỗi khi gặp bóng sáng `p`, nếu `p - prev - 1 == k` → ngày hiện tại là đáp án.

Đây là cách "đúng theo định nghĩa", rất khó sai — ta dùng nó làm **trọng tài** để kiểm tra các cách nhanh hơn.

In [8]:
def k_empty_brute(bulbs, k):
    n = len(bulbs)
    state = [False] * (n + 1)              # state[p]: vị trí p đang sáng?
    for day, x in enumerate(bulbs, start=1):
        state[x] = True
        prev = None                        # vị trí sáng gần nhất bên trái khi quét
        for p in range(1, n + 1):
            if state[p]:
                if prev is not None and p - prev - 1 == k:
                    return day
                prev = p
    return -1

print(k_empty_brute([1, 3, 2], 1))          # 2
print(k_empty_brute([1, 2, 3], 1))          # -1
print(k_empty_brute([3, 1, 5, 4, 2], 1))    # 2
print(k_empty_brute([1, 5, 3, 2, 4], 1))    # 3
print(k_empty_brute([2, 1, 3], 0))          # 2 (hai bóng sáng kề nhau)

2
-1
2
3
2


**Độ phức tạp.** `n` ngày × quét `n` vị trí = `O(n²)` thời gian, `O(n)` bộ nhớ. Với `n = 2·10^4` là `4·10^8` thao tác — quá chậm với Python.

**Nút thắt.** Mỗi ngày ta quét lại **toàn bộ** hàng, trong khi chỉ có một bóng thay đổi. Quan sát 4.3 nói rằng chỉ cần nhìn hai hàng xóm của bóng mới.

## 6. Bước đột phá

**Insight 1 (theo thời gian):** Khi bật `x`, chỉ có hai cặp mới: `(L, x)` và `(x, R)` với `L`, `R` là bóng sáng gần nhất bên trái/phải. Vậy:

> **Nếu** `x - L - 1 == k` **hoặc** `R - x - 1 == k` **thì** hôm nay là đáp án.

Cài đặt ngây thơ: nhìn sang trái tối đa `k + 1` ô để tìm `L` (tương tự bên phải) → `O(n·k)`. Tốt hơn: giữ **danh sách vị trí sáng đã sắp xếp** và dùng `bisect` → Cách 2.

**Insight 2 (đổi góc nhìn):** Lập `days[p]` = ngày bật vị trí `p`. Một cặp vị trí `(left, right)` với `right = left + k + 1` trở thành "cặp hợp lệ" vào ngày `max(days[left], days[right])` **nếu và chỉ nếu** mọi vị trí bên trong được bật **muộn hơn** ngày đó:

> `days[i] > max(days[left], days[right])` với mọi `left < i < right`.

Đáp án = **min** của `max(days[left], days[right])` trên mọi cửa sổ hợp lệ. Đây là bài cửa sổ độ dài cố định trên mảng tĩnh → Cách 3, `O(n)`.

**Insight 3 (chia xô):** Hai vị trí cách nhau đúng `k + 1` thì nằm ở **hai xô liền nhau** nếu chia hàng thành các xô dài `k + 1`. Và nếu không có bóng sáng nào giữa chúng, thì bóng trái là **max** xô của nó, bóng phải là **min** xô kế tiếp. Chỉ cần lưu min/max mỗi xô → Cách 4, `O(n)`.

Kiểm tra nhanh Insight 2 trên ví dụ 1:

In [9]:
bulbs, k = [1, 3, 2], 1
n = len(bulbs)
days = [0] * n
for d, x in enumerate(bulbs, 1):
    days[x - 1] = d
print("days =", days)
for left in range(n - k - 1):
    right = left + k + 1
    inside = days[left + 1:right]
    ok = all(v > max(days[left], days[right]) for v in inside)
    print(f"cửa sổ vị trí ({left + 1}, {right + 1}): ngày hai đầu {days[left]}, {days[right]}, bên trong {inside} -> {'hợp lệ, ngày ' + str(max(days[left], days[right])) if ok else 'không hợp lệ'}")

days = [1, 3, 2]
cửa sổ vị trí (1, 3): ngày hai đầu 1, 2, bên trong [3] -> hợp lệ, ngày 2


## 7. Các cách tiếp theo

Tổng cộng 4 cách:

| Cách | Ý tưởng | Time |
|---|---|---|
| 1 | Brute force quét cả hàng mỗi ngày | `O(n²)` |
| 2 | Danh sách vị trí sáng đã sắp + `bisect` tìm hàng xóm | `O(n log n)` tìm + `O(n²)` chèn tệ nhất (thực tế rất nhanh) |
| 3 | `days[]` + cửa sổ trượt độ dài `k + 2` | `O(n)` |
| 4 | Chia xô kích thước `k + 1`, lưu min/max | `O(n)` |

### 7.1. Cách 2 — Danh sách đã sắp + `bisect`

**Giải thích.** Giữ `on` là danh sách các vị trí đang sáng, luôn sắp tăng. Mỗi ngày bật `x`:

1. `i = bisect_left(on, x)` → hàng xóm trái `on[i-1]`, hàng xóm phải `on[i]`.
2. Kiểm tra hai khoảng cách.
3. Chèn `x` vào vị trí `i`.

In [10]:
def k_empty_bisect(bulbs, k):
    on = []                                    # vị trí đang sáng, sắp tăng
    for day, x in enumerate(bulbs, start=1):
        i = bisect_left(on, x)
        if i > 0 and x - on[i - 1] - 1 == k:   # cặp (hàng xóm trái, x)
            return day
        if i < len(on) and on[i] - x - 1 == k: # cặp (x, hàng xóm phải)
            return day
        on.insert(i, x)
    return -1

for b, k in [([1, 3, 2], 1), ([1, 2, 3], 1), ([3, 1, 5, 4, 2], 1), ([1, 5, 3, 2, 4], 1), ([2, 1, 3], 0)]:
    print(b, k, "->", k_empty_bisect(b, k))

[1, 3, 2] 1 -> 2
[1, 2, 3] 1 -> -1
[3, 1, 5, 4, 2] 1 -> 2
[1, 5, 3, 2, 4] 1 -> 3
[2, 1, 3] 0 -> 2


**Code nộp LeetCode (Cách 2):**

```python
from bisect import bisect_left

class Solution:
    def kEmptySlots(self, bulbs: List[int], k: int) -> int:
        on = []
        for day, x in enumerate(bulbs, 1):
            i = bisect_left(on, x)
            if i > 0 and x - on[i - 1] - 1 == k:
                return day
            if i < len(on) and on[i] - x - 1 == k:
                return day
            on.insert(i, x)
        return -1
```

**Độ phức tạp.** Tìm kiếm `O(log n)` mỗi ngày; `list.insert` là `O(n)` tệ nhất → tổng `O(n²)` tệ nhất, nhưng `insert` được cài bằng `memmove` trong C nên với `n = 2·10^4` vẫn rất nhanh. Trong Java/C++ người ta dùng `TreeSet`/`std::set` để có `O(n log n)` thật sự. Bộ nhớ `O(n)`.

### 7.2. Cách 3 — `days[]` + cửa sổ trượt `O(n)`

**Giải thích.** Xét cửa sổ `(left, right = left + k + 1)` trên mảng `days` (chỉ số 0). Dùng con trỏ `i` chạy bên trong cửa sổ:

- Nếu `days[i] < max(days[left], days[right])` (một bóng bên trong sáng **trước** một đầu mút) → cửa sổ hỏng. Hơn nữa, mọi cửa sổ có `left'` nằm giữa `left` và `i` cũng hỏng (vì chúng chứa `i` bên trong và `i` sáng sớm hơn `left'`). Vậy nhảy thẳng `left = i`.
- Nếu `i` chạy tới `right` mà không hỏng → cửa sổ hợp lệ, cập nhật đáp án `max(days[left], days[right])`, rồi nhảy `left = right` (các cửa sổ có `left'` ở giữa đều chứa `right` bên trong, mà `right` sáng sớm hơn chúng → hỏng).

Mỗi lần nhảy `left` tiến lên ít nhất một ô và `i` không bao giờ lùi quá `left + 1` → `O(n)`.

In [11]:
def k_empty_window(bulbs, k):
    n = len(bulbs)
    days = [0] * n
    for day, x in enumerate(bulbs, start=1):
        days[x - 1] = day
    INF = float('inf')
    ans = INF
    left, right = 0, k + 1
    i = left + 1
    while right < n:
        if i == right:                                   # cửa sổ hợp lệ
            ans = min(ans, max(days[left], days[right]))
            left, right = right, right + k + 1
            i = left + 1
        elif days[i] < max(days[left], days[right]):     # cửa sổ hỏng tại i
            left, right = i, i + k + 1
            i = left + 1
        else:
            i += 1
    return ans if ans < INF else -1

for b, k in [([1, 3, 2], 1), ([1, 2, 3], 1), ([3, 1, 5, 4, 2], 1), ([1, 5, 3, 2, 4], 1), ([2, 1, 3], 0)]:
    print(b, k, "->", k_empty_window(b, k))

[1, 3, 2] 1 -> 2
[1, 2, 3] 1 -> -1
[3, 1, 5, 4, 2] 1 -> 2
[1, 5, 3, 2, 4] 1 -> 3
[2, 1, 3] 0 -> 2


Giải thích từng dòng:

- `days[x - 1] = day` — đảo mảng (chỉ số 0 cho tiện).
- `left, right = 0, k + 1` — cửa sổ đầu tiên; nếu `k + 1 >= n` vòng `while` không chạy → `-1` (không đủ chỗ cho `k` bóng ở giữa).
- `i == right` — mọi ô bên trong đã kiểm tra xong, tất cả đều sáng muộn hơn hai đầu.
- `days[i] < max(...)` — có ô bên trong sáng sớm hơn ít nhất một đầu → hỏng; `i` trở thành `left` mới (ứng viên tốt vì nó sáng sớm).
- Khi `k = 0`: `i = left + 1 = right` ngay → mọi cặp kề đều hợp lệ, đáp án là min của `max(days[p], days[p+1])`.

**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def kEmptySlots(self, bulbs: List[int], k: int) -> int:
        n = len(bulbs)
        days = [0] * n
        for day, x in enumerate(bulbs, 1):
            days[x - 1] = day
        ans = float('inf')
        left, right, i = 0, k + 1, 1
        while right < n:
            if i == right:
                ans = min(ans, max(days[left], days[right]))
                left, right = right, right + k + 1
                i = left + 1
            elif days[i] < max(days[left], days[right]):
                left, right = i, i + k + 1
                i = left + 1
            else:
                i += 1
        return ans if ans < float('inf') else -1
```

**Độ phức tạp.** `O(n)` thời gian, `O(n)` bộ nhớ cho `days`.

### 7.3. Cách 4 — Chia xô kích thước `k + 1`

**Giải thích.** Chia các vị trí `0..n-1` thành xô `b = p // (k + 1)`. Mỗi xô lưu `lo[b]` (vị trí sáng nhỏ nhất) và `hi[b]` (lớn nhất).

Khi bật vị trí `p` trong xô `b`:

- **Nếu** `p` trở thành **min mới** của xô `b` → bóng sáng gần nhất bên trái của `p` chính là `hi[b-1]` (vì trong xô `b` không có bóng sáng nào nhỏ hơn `p`, và xô `b-1` nằm ngay bên trái). Nếu `hi[b-1] == p - (k+1)` → đáp án.
- **Nếu** `p` **không** là min của xô `b` → có bóng sáng `q < p` cùng xô, `p - q < k + 1` → cặp bên trái có khoảng cách quá ngắn, không cần xét.
- Đối xứng với bên phải: nếu `p` thành **max mới** của xô `b`, so với `lo[b+1]`.

In [12]:
def k_empty_bucket(bulbs, k):
    n = len(bulbs)
    size = k + 1
    nb = n // size + 1                    # số xô
    INF = float('inf')
    lo = [INF] * nb                       # vị trí sáng nhỏ nhất mỗi xô
    hi = [-INF] * nb                      # vị trí sáng lớn nhất mỗi xô
    for day, x in enumerate(bulbs, start=1):
        p = x - 1
        b = p // size
        if p < lo[b]:
            lo[b] = p
            if b > 0 and hi[b - 1] == p - size:
                return day
        if p > hi[b]:
            hi[b] = p
            if b + 1 < nb and lo[b + 1] == p + size:
                return day
    return -1

for b, k in [([1, 3, 2], 1), ([1, 2, 3], 1), ([3, 1, 5, 4, 2], 1), ([1, 5, 3, 2, 4], 1), ([2, 1, 3], 0)]:
    print(b, k, "->", k_empty_bucket(b, k))

[1, 3, 2] 1 -> 2
[1, 2, 3] 1 -> -1
[3, 1, 5, 4, 2] 1 -> 2
[1, 5, 3, 2, 4] 1 -> 3
[2, 1, 3] 0 -> 2


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def kEmptySlots(self, bulbs: List[int], k: int) -> int:
        n, size = len(bulbs), k + 1
        nb = n // size + 1
        lo = [float('inf')] * nb
        hi = [float('-inf')] * nb
        for day, x in enumerate(bulbs, 1):
            p = x - 1
            b = p // size
            if p < lo[b]:
                lo[b] = p
                if b > 0 and hi[b - 1] == p - size:
                    return day
            if p > hi[b]:
                hi[b] = p
                if b + 1 < nb and lo[b + 1] == p + size:
                    return day
        return -1
```

**Độ phức tạp.** `O(n)` thời gian (mỗi ngày `O(1)`), `O(n / (k+1))` bộ nhớ cho các xô. Thêm một điểm cộng: dừng ngay tại ngày đầu tiên tìm thấy (như Cách 2), không cần duyệt hết như Cách 3.

## 8. TRACE từng bước

### 8.1. Bảng trạng thái Cách 2 cho `bulbs = [1, 5, 3, 2, 4]`, `k = 1`

| Ngày | x | `on` trước | `i` | trái | phải | `x - trái - 1` | `phải - x - 1` | Kết luận |
|---|---|---|---|---|---|---|---|---|
| 1 | 1 | `[]` | 0 | — | — | — | — | chèn |
| 2 | 5 | `[1]` | 1 | 1 | — | 3 | — | chèn |
| 3 | 3 | `[1, 5]` | 1 | 1 | 5 | **1** | 1 | **trả về 3** |

### 8.2. `trace_bisect`

In [13]:
def trace_bisect(bulbs, k):
    on = []
    print(f"bulbs={bulbs}, k={k}")
    for day, x in enumerate(bulbs, start=1):
        i = bisect_left(on, x)
        L = on[i - 1] if i > 0 else None
        R = on[i] if i < len(on) else None
        gl = x - L - 1 if L is not None else None
        gr = R - x - 1 if R is not None else None
        print(f"  ngày {day}: x={x}, on={on}, i={i}, trái={L}, phải={R}, khe trái={gl}, khe phải={gr}")
        if gl == k or gr == k:
            print(f"  => tìm thấy, trả về {day}")
            return day
        on.insert(i, x)
    print("  => không có, trả về -1")
    return -1

trace_bisect([1, 5, 3, 2, 4], 1)
trace_bisect([1, 2, 3], 1)

bulbs=[1, 5, 3, 2, 4], k=1
  ngày 1: x=1, on=[], i=0, trái=None, phải=None, khe trái=None, khe phải=None
  ngày 2: x=5, on=[1], i=1, trái=1, phải=None, khe trái=3, khe phải=None
  ngày 3: x=3, on=[1, 5], i=1, trái=1, phải=5, khe trái=1, khe phải=1
  => tìm thấy, trả về 3
bulbs=[1, 2, 3], k=1
  ngày 1: x=1, on=[], i=0, trái=None, phải=None, khe trái=None, khe phải=None
  ngày 2: x=2, on=[1], i=1, trái=1, phải=None, khe trái=0, khe phải=None
  ngày 3: x=3, on=[1, 2], i=2, trái=2, phải=None, khe trái=0, khe phải=None
  => không có, trả về -1


-1

### 8.3. `trace_window` — cửa sổ trượt trên `days`

Ví dụ `bulbs = [6, 5, 8, 9, 7, 1, 10, 2, 3, 4]`, `k = 2`. Hàm in `left`, `right`, `i` và lý do mỗi lần nhảy.

In [14]:
def trace_window(bulbs, k):
    n = len(bulbs)
    days = [0] * n
    for day, x in enumerate(bulbs, start=1):
        days[x - 1] = day
    print("days (theo vị trí 1..n):", days)
    INF = float('inf')
    ans = INF
    left, right, i = 0, k + 1, 1
    while right < n:
        if i == right:
            cand = max(days[left], days[right])
            ans = min(ans, cand)
            print(f"  cửa sổ vị trí [{left + 1}..{right + 1}] HỢP LỆ, ngày {cand}, ans={ans}")
            left, right = right, right + k + 1
            i = left + 1
        elif days[i] < max(days[left], days[right]):
            print(f"  cửa sổ [{left + 1}..{right + 1}] hỏng tại vị trí {i + 1} (ngày {days[i]} < max({days[left]}, {days[right]})) -> left = {i + 1}")
            left, right = i, i + k + 1
            i = left + 1
        else:
            i += 1
    res = ans if ans < INF else -1
    print("  kết quả:", res)
    return res

trace_window([6, 5, 8, 9, 7, 1, 10, 2, 3, 4], 2)

days (theo vị trí 1..n): [6, 8, 9, 10, 2, 1, 5, 3, 4, 7]
  cửa sổ [1..4] hỏng tại vị trí 2 (ngày 8 < max(6, 10)) -> left = 2
  cửa sổ vị trí [2..5] HỢP LỆ, ngày 8, ans=8
  cửa sổ [5..8] hỏng tại vị trí 6 (ngày 1 < max(2, 3)) -> left = 6
  cửa sổ [6..9] hỏng tại vị trí 8 (ngày 3 < max(1, 4)) -> left = 8
  kết quả: 8


8

### 8.4. `trace_bucket` — min/max các xô

In [15]:
def trace_bucket(bulbs, k):
    n, size = len(bulbs), k + 1
    nb = n // size + 1
    INF = float('inf')
    lo, hi = [INF] * nb, [-INF] * nb
    fmt = lambda arr: ['-' if abs(v) == INF else v + 1 for v in arr]   # in theo vị trí 1..n
    print(f"n={n}, k={k}, size={size}, số xô={nb}")
    for day, x in enumerate(bulbs, start=1):
        p = x - 1
        b = p // size
        msg = ""
        if p < lo[b]:
            lo[b] = p
            if b > 0 and hi[b - 1] == p - size:
                msg = f"min mới xô {b}, max xô {b-1} = {hi[b-1] + 1} cách đúng {size} -> TRẢ VỀ {day}"
        if not msg and p > hi[b]:
            hi[b] = p
            if b + 1 < nb and lo[b + 1] == p + size:
                msg = f"max mới xô {b}, min xô {b+1} = {lo[b+1] + 1} cách đúng {size} -> TRẢ VỀ {day}"
        print(f"  ngày {day}: bật {x} (xô {b}) lo={fmt(lo)} hi={fmt(hi)} {msg}")
        if msg:
            return day
    print("  -> -1")
    return -1

trace_bucket([6, 5, 8, 9, 7, 1, 10, 2, 3, 4], 2)

n=10, k=2, size=3, số xô=4
  ngày 1: bật 6 (xô 1) lo=['-', 6, '-', '-'] hi=['-', 6, '-', '-'] 
  ngày 2: bật 5 (xô 1) lo=['-', 5, '-', '-'] hi=['-', 6, '-', '-'] 
  ngày 3: bật 8 (xô 2) lo=['-', 5, 8, '-'] hi=['-', 6, 8, '-'] 
  ngày 4: bật 9 (xô 2) lo=['-', 5, 8, '-'] hi=['-', 6, 9, '-'] 
  ngày 5: bật 7 (xô 2) lo=['-', 5, 7, '-'] hi=['-', 6, 9, '-'] 
  ngày 6: bật 1 (xô 0) lo=[1, 5, 7, '-'] hi=[1, 6, 9, '-'] 
  ngày 7: bật 10 (xô 3) lo=[1, 5, 7, 10] hi=[1, 6, 9, 10] 
  ngày 8: bật 2 (xô 0) lo=[1, 5, 7, 10] hi=[2, 6, 9, 10] max mới xô 0, min xô 1 = 5 cách đúng 3 -> TRẢ VỀ 8


8

## 9. Code nộp LeetCode

Chọn **Cách 4 (chia xô)**: `O(n)`, dừng sớm, code ngắn. (Cách 2 và Cách 3 cũng đều được chấp nhận.)

In [16]:
from typing import List

class Solution:
    def kEmptySlots(self, bulbs: List[int], k: int) -> int:
        n, size = len(bulbs), k + 1
        nb = n // size + 1
        lo = [float('inf')] * nb
        hi = [float('-inf')] * nb
        for day, x in enumerate(bulbs, 1):
            p = x - 1
            b = p // size
            if p < lo[b]:
                lo[b] = p
                if b > 0 and hi[b - 1] == p - size:
                    return day
            if p > hi[b]:
                hi[b] = p
                if b + 1 < nb and lo[b + 1] == p + size:
                    return day
        return -1

assert Solution().kEmptySlots([1, 3, 2], 1) == 2
assert Solution().kEmptySlots([1, 2, 3], 1) == -1
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bổ đề (chỉ cặp mới).** Nếu ngày `d` là ngày đầu tiên có cặp hợp lệ `(a, b)`, thì một trong `a, b` được bật đúng ngày `d`. Thật vậy, nếu cả hai sáng từ trước và vào ngày `d-1` giữa chúng đã đều tắt (vì ngày `d` vẫn tắt và bóng không bao giờ tắt lại) thì ngày `d-1` đã có cặp hợp lệ — mâu thuẫn. Hơn nữa `a`, `b` phải là hai bóng sáng **kề nhau** trong dãy sáng. Vậy chỉ cần kiểm tra (hàng xóm trái, x) và (x, hàng xóm phải).

**Cách 2** kiểm tra đúng hai cặp đó nhờ `bisect` → đúng theo bổ đề.

**Cách 3.** Cặp `(left, right = left + k + 1)` trở thành hợp lệ vào ngày `max(days[left], days[right])` khi và chỉ khi mọi ô trong có ngày lớn hơn. Đáp án = min trên các cửa sổ hợp lệ. Thuật toán chỉ bỏ qua các cửa sổ chắc chắn hỏng (lập luận nhảy `left` ở mục 7.2), nên không bỏ sót cửa sổ hợp lệ nào.

**Cách 4.** Cặp hợp lệ `(a, b)` có `b - a = k + 1 = size` nên `b // size = a // size + 1` (hai xô liền nhau). Vào ngày cặp hình thành, không có bóng sáng nào giữa `a` và `b`, nên `a` là max xô của nó và `b` là min xô của nó. Bóng bật hôm đó (giả sử là `b`) vừa trở thành min mới → nhánh `p < lo[b]` kiểm tra đúng `hi[b-1] == p - size`. Ngược lại, khi thuật toán trả về, `hi[b-1]` và `p` cách nhau `size`, không có bóng sáng nào trong xô `b` nhỏ hơn `p` và không có bóng sáng nào trong xô `b-1` lớn hơn `hi[b-1]` → khoảng giữa toàn tắt → cặp hợp lệ thật.

## 11. Phân tích Time/Space

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n²)` | `O(n)` | quá chậm khi `n = 2·10^4` trong Python |
| 2. Bisect | `O(n log n)` tìm + `O(n²)` chèn tệ nhất | `O(n)` | `insert` bằng C nên thực tế nhanh |
| 3. Cửa sổ trượt | `O(n)` | `O(n)` | phải duyệt hết mảng |
| 4. Chia xô | `O(n)` | `O(n/(k+1))` | dừng sớm |

Đo thử trên `n = 20000` (hoán vị ngẫu nhiên, vài giá trị `k`):

In [17]:
import random, time

random.seed(683)
big = list(range(1, 20001))
random.shuffle(big)
for k in [0, 5, 100, 5000]:
    row = []
    res = set()
    for name, fn in [("bisect", k_empty_bisect), ("window", k_empty_window), ("bucket", k_empty_bucket)]:
        t0 = time.perf_counter()
        res.add(fn(big, k))
        row.append(f"{name}={time.perf_counter() - t0:.3f}s")
    assert len(res) == 1
    print(f"k={k:>5}: đáp án {res.pop():>6} | " + ", ".join(row))

k=    0: đáp án    216 | bisect=0.000s, window=0.003s, bucket=0.000s
k=    5: đáp án    102 | bisect=0.000s, window=0.003s, bucket=0.000s
k=  100: đáp án     60 | bisect=0.000s, window=0.002s, bucket=0.000s
k= 5000: đáp án     -1 | bisect=0.022s, window=0.002s, bucket=0.001s


## 12. Bộ test

1. Ví dụ đề.
2. Edge case: `n = 1`, `k = 0`, `k >= n`, bật từ trái sang phải (không bao giờ có khe), bật xen kẽ.
3. Đối chiếu ngẫu nhiên có seed: mọi cách so với brute force với `n` nhỏ và mọi `k` từ 0 tới `n`.

In [18]:
METHODS = [k_empty_bisect, k_empty_window, k_empty_bucket, lambda b, k: Solution().kEmptySlots(b, k)]

def check(bulbs, k, expected=None):
    ref = k_empty_brute(bulbs, k)
    for f in METHODS:
        got = f(bulbs, k)
        assert got == ref, (bulbs, k, got, ref)
    if expected is not None:
        assert ref == expected, (bulbs, k, ref, expected)
    return ref

# 1. Ví dụ đề
check([1, 3, 2], 1, 2)
check([1, 2, 3], 1, -1)

# 2. Edge case
check([1], 0, -1)                      # chỉ một bóng
check([2, 1], 0, 2)                    # k = 0: hai bóng kề nhau
check([1, 2], 1, -1)                   # k quá lớn so với n
check([1, 2, 3, 4, 5], 0, 2)           # bật trái sang phải, k = 0
check([1, 2, 3, 4, 5], 2, -1)          # không bao giờ có khe
check([1, 3, 5, 2, 4], 1, 2)           # xen kẽ
check([5, 1, 4, 2, 3], 3, 2)           # hai đầu mút trước tiên
check([3, 1, 5, 4, 2], 1, 2)
check([6, 5, 8, 9, 7, 1, 10, 2, 3, 4], 2, 8)
print("Ví dụ + edge case: OK")

# 3. Ngẫu nhiên có seed
random.seed(2024)
cnt = 0
for _ in range(1500):
    n = random.randint(1, 12)
    b = list(range(1, n + 1))
    random.shuffle(b)
    for k in range(0, n + 1):
        check(b, k)
        cnt += 1
print(f"{cnt} bộ (bulbs, k) ngẫu nhiên: mọi cách khớp brute force")

Ví dụ + edge case: OK


11225 bộ (bulbs, k) ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1: Hiểu nhầm `bulbs[i]` là "ngày của vị trí i"

In [19]:
def k_empty_wrong_meaning(bulbs, k):
    days = bulbs[:]                        # SAI: coi bulbs chính là days
    n = len(days)
    best = float('inf')
    for left in range(n - k - 1):
        right = left + k + 1
        if all(days[i] > max(days[left], days[right]) for i in range(left + 1, right)):
            best = min(best, max(days[left], days[right]))
    return best if best < float('inf') else -1

b = [6, 5, 8, 9, 7, 1, 10, 2, 3, 4]
print("sai:", k_empty_wrong_meaning(b, 2), "| đúng:", k_empty_brute(b, 2))

sai: 7 | đúng: 8


**Vì sao sai:** đề cho "ngày → vị trí". Phải đảo: `days[bulbs[i] - 1] = i + 1`. Hai mảng chỉ trùng nhau khi hoán vị là tự nghịch đảo (ví dụ `[1, 3, 2]` — chính ví dụ đề, nên lỗi này rất dễ lọt qua!).

In [20]:
def k_empty_fixed_meaning(bulbs, k):
    n = len(bulbs)
    days = [0] * n
    for d, x in enumerate(bulbs, 1):
        days[x - 1] = d                    # ĐÚNG: đảo mảng
    best = float('inf')
    for left in range(n - k - 1):
        right = left + k + 1
        if all(days[i] > max(days[left], days[right]) for i in range(left + 1, right)):
            best = min(best, max(days[left], days[right]))
    return best if best < float('inf') else -1

print("ví dụ đề vẫn đúng với code sai:", k_empty_wrong_meaning([1, 3, 2], 1))
print("đã sửa:", k_empty_fixed_meaning(b, 2))
assert k_empty_fixed_meaning(b, 2) == k_empty_brute(b, 2)

ví dụ đề vẫn đúng với code sai: 2
đã sửa: 8


### Lỗi 2: Lệch một — dùng `x - L == k` thay cho `x - L - 1 == k`

In [21]:
def k_empty_off_by_one(bulbs, k):
    on = []
    for day, x in enumerate(bulbs, 1):
        i = bisect_left(on, x)
        if i > 0 and x - on[i - 1] == k:          # SAI: đó là khoảng cách, không phải số bóng ở giữa
            return day
        if i < len(on) and on[i] - x == k:
            return day
        on.insert(i, x)
    return -1

print("sai:", k_empty_off_by_one([1, 3, 2], 1), "| đúng:", k_empty_bisect([1, 3, 2], 1))

sai: 3 | đúng: 2


**Vì sao sai:** giữa vị trí `a` và `b` có `b - a - 1` bóng. "Đúng `k` bóng ở giữa" ⇔ `b - a == k + 1`. Code đúng là `k_empty_bisect`.

### Lỗi 3: Chỉ so với bóng sáng **gần nhất theo thời gian** hoặc chỉ nhìn đúng ô `x ± (k+1)`

In [22]:
def k_empty_wrong_lookup(bulbs, k):
    n = len(bulbs)
    on = [False] * (n + 2)
    for day, x in enumerate(bulbs, 1):
        on[x] = True
        # SAI: chỉ kiểm tra ô cách k+1 có sáng không, quên kiểm tra k ô ở giữa đều tắt
        if x - k - 1 >= 1 and on[x - k - 1]:
            return day
        if x + k + 1 <= n and on[x + k + 1]:
            return day
    return -1

b = [1, 2, 3]
print("sai:", k_empty_wrong_lookup(b, 1), "| đúng:", k_empty_brute(b, 1))

sai: 3 | đúng: -1


**Vì sao sai:** với `[1, 2, 3]`, ngày 3 bật vị trí 3, ô `3 - 2 = 1` sáng nhưng ô 2 ở giữa **cũng sáng** → không hợp lệ. Phải đảm bảo không có bóng sáng ở giữa: dùng hàng xóm sáng gần nhất (`bisect`) hoặc kiểm tra cả đoạn giữa:

In [23]:
def k_empty_local_check(bulbs, k):
    n = len(bulbs)
    on = [False] * (n + 2)
    for day, x in enumerate(bulbs, 1):
        on[x] = True
        L = x - k - 1
        if L >= 1 and on[L] and not any(on[L + 1:x]):           # ĐÚNG: kiểm tra đoạn giữa
            return day
        R = x + k + 1
        if R <= n and on[R] and not any(on[x + 1:R]):
            return day
    return -1

print("đã sửa:", k_empty_local_check([1, 2, 3], 1))
random.seed(13)
for _ in range(300):
    n = random.randint(1, 10)
    bb = random.sample(range(1, n + 1), n)
    for k in range(n + 1):
        assert k_empty_local_check(bb, k) == k_empty_brute(bb, k)
print("k_empty_local_check (O(n·k)) khớp brute force")

đã sửa: -1
k_empty_local_check (O(n·k)) khớp brute force


### Lỗi 4: Cửa sổ trượt quên trả về **min** mà trả về cửa sổ hợp lệ **đầu tiên theo vị trí**

Cửa sổ hợp lệ đầu tiên từ trái sang phải chưa chắc có ngày nhỏ nhất (ví dụ cửa sổ bên phải hình thành sớm hơn). Phải duyệt hết và lấy `min`, như `k_empty_window`.

## 14. Mẹo & nhận dạng pattern

- **"Mỗi ngày bật/thêm một phần tử, hỏi ngày đầu tiên thoả điều kiện"** → hoặc mô phỏng + cấu trúc dữ liệu có thứ tự (`bisect`, `SortedList`, `TreeSet`), hoặc **đảo mảng** thành `time[pos]` rồi giải bài tĩnh.
- **"Hai phần tử kề nhau cách đúng D"** → chia xô kích thước `D` (giống mẹo của *Maximum Gap* — LeetCode 164 — và *Contains Duplicate III* — LeetCode 220).
- **"Mọi phần tử bên trong lớn hơn hai đầu mút"** → cửa sổ trượt với bước nhảy (hoặc deque min trên cửa sổ độ dài cố định).
- Khi đề có ví dụ là **hoán vị tự nghịch đảo**, cẩn thận: ví dụ không phân biệt được "ngày→vị trí" và "vị trí→ngày".

## 15. Biến thể

1. **Ngày cuối cùng** còn tồn tại cặp hợp lệ? (Dùng `days` + cửa sổ, lấy cửa sổ hợp lệ và chú ý cặp bị phá khi một bóng bên trong bật.)
2. **"Ít nhất `k` bóng tắt"** ở giữa thay vì đúng `k`: với Cách 2 đổi `==` thành `>=`.
3. **Đếm số cửa sổ** từng hợp lệ ở bất kỳ thời điểm nào: Cách 3 đếm số lần `i == right`.
4. **Bóng có thể tắt lại** (thao tác bật/tắt) → cần cấu trúc động hai chiều (SortedList), cách cửa sổ không còn dùng được.

Ví dụ biến thể 2 ("ít nhất `k`") và 3 ("đếm cửa sổ hợp lệ"):

In [24]:
def at_least_k(bulbs, k):
    on = []
    for day, x in enumerate(bulbs, 1):
        i = bisect_left(on, x)
        if (i > 0 and x - on[i - 1] - 1 >= k) or (i < len(on) and on[i] - x - 1 >= k):
            return day
        on.insert(i, x)
    return -1

def count_valid_windows(bulbs, k):
    n = len(bulbs)
    days = [0] * n
    for d, x in enumerate(bulbs, 1):
        days[x - 1] = d
    return sum(all(days[i] > max(days[l], days[l + k + 1]) for i in range(l + 1, l + k + 1))
               for l in range(n - k - 1))

print(at_least_k([1, 5, 3, 2, 4], 2))           # ngày 2: vị trí 1 và 5, ở giữa 3 bóng ≥ 2
print(count_valid_windows([6, 5, 8, 9, 7, 1, 10, 2, 3, 4], 2))

2
1


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Mô phỏng, quét cả hàng mỗi ngày | `O(n²)` | `O(n)` | Trọng tài khi test; `n` rất nhỏ |
| 2. Bisect | Danh sách sáng đã sắp; chỉ xét hai hàng xóm của bóng mới | `O(n log n)` (+ chèn) | `O(n)` | Tự nhiên nhất, dễ mở rộng (bật/tắt, "ít nhất k") |
| 3. Cửa sổ trượt | `days[pos]`; cửa sổ hợp lệ khi bên trong đều sáng muộn hơn | `O(n)` | `O(n)` | Khi muốn `O(n)` chắc chắn, bài tĩnh |
| 4. Chia xô | Xô `k+1`; chỉ so max xô trái với min xô phải | `O(n)` | `O(n/(k+1))` | `O(n)` + dừng sớm, code ngắn |

## 17. Bài tập

**Bài 1 (đoán output).** `kEmptySlots([2, 4, 1, 3], 1)`, `kEmptySlots([4, 1, 3, 2], 0)`, `kEmptySlots([1, 4, 2, 3], 2)`?

**Bài 2 (tính tay).** Lập mảng `days` và bảng nhảy cửa sổ (như 8.3) cho `bulbs = [3, 6, 1, 5, 2, 4]`, `k = 1`.

**Bài 3 (điền TODO).** Hoàn thiện `invert(bulbs)` trả về `days` (chỉ số 0).

**Bài 4 (tự viết lại).** Viết lại `my_k_empty(bulbs, k)` bằng chia xô.

**Bài 5 (mini challenge).** Viết `last_day_with_pair(bulbs, k)`: ngày **lớn nhất** `d` mà cuối ngày `d` vẫn tồn tại một cặp hợp lệ (hoặc `-1`). Gợi ý: cặp `(a, b)` hợp lệ trong khoảng ngày `[max(days[a], days[b]), min(days bên trong) - 1]`.

In [25]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def invert(bulbs):
    # TODO: days = [0]*n; với (day, x) trong enumerate(bulbs, 1): days[x-1] = day
    pass

# Bài 4 — TODO
def my_k_empty(bulbs, k):
    # TODO: size = k+1; lo/hi cho mỗi xô; khi p là min mới so với hi[b-1]; khi p là max mới so với lo[b+1]
    pass

# Bài 5 — TODO
def last_day_with_pair(bulbs, k):
    # TODO: với mỗi left, right = left+k+1: start = max(days[left], days[right]);
    #       end = min(days bên trong) - 1 (nếu không có ô trong (k=0) thì end = n);
    #       nếu start <= end, cập nhật đáp án = max(đáp án, end)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: mô phỏng bằng `show_days`.
- Mức 2: `[2,4,1,3], k=1`: ngày 2 sáng {2, 4}, giữa là 3 tắt.
- Mức 3: đáp án `2`; `[4,1,3,2], k=0`: ngày 3 sáng {1,3,4}, 3 và 4 kề → `3`; `[1,4,2,3], k=2`: ngày 2 sáng {1,4}, giữa 2 bóng → `2`.

**Bài 2.**
- Mức 1: `days[x-1] = ngày`.
- Mức 2: `days = [3, 5, 1, 6, 4, 2]`.
- Mức 3: cửa sổ `[1..3]`: bên trong vị trí 2 có ngày 5 > max(3, 1) → hợp lệ, ngày 3; nhảy left = 3: cửa sổ `[3..5]`: vị trí 4 có ngày 6 > max(1, 4) → hợp lệ, ngày 4; `[5..7]` vượt n → dừng. Đáp án `min(3, 4) = 3` (kiểm tra bằng mô phỏng: ngày 3 sáng {1, 3, 6}, giữa 1 và 3 là bóng 2 đang tắt).

**Bài 3.**
- Mức 1: tạo mảng `n` số 0.
- Mức 2: duyệt `enumerate(bulbs, 1)`.
- Mức 3: gán `days[x - 1] = day`.

**Bài 4.**
- Mức 1: `nb = n // size + 1`.
- Mức 2: `lo` khởi tạo `inf`, `hi` khởi tạo `-inf`.
- Mức 3: hai nhánh `if` độc lập (một bóng có thể vừa là min vừa là max của xô đang rỗng).

**Bài 5.**
- Mức 1: cặp `(a, b)` bắt đầu hợp lệ khi bóng thứ hai trong hai đầu bật.
- Mức 2: cặp bị phá khi một bóng bên trong bật đầu tiên.
- Mức 3: đối chiếu với brute force mô phỏng mỗi ngày.

## 19. Đáp án tham khảo

In [26]:
# Bài 1
assert Solution().kEmptySlots([2, 4, 1, 3], 1) == 2
assert Solution().kEmptySlots([4, 1, 3, 2], 0) == 3
assert Solution().kEmptySlots([1, 4, 2, 3], 2) == 2

# Bài 2
def invert_ans(bulbs):
    days = [0] * len(bulbs)
    for day, x in enumerate(bulbs, 1):
        days[x - 1] = day
    return days

assert invert_ans([3, 6, 1, 5, 2, 4]) == [3, 5, 1, 6, 4, 2]
assert k_empty_window([3, 6, 1, 5, 2, 4], 1) == 3 == k_empty_brute([3, 6, 1, 5, 2, 4], 1)

# Bài 3
assert invert_ans([1, 3, 2]) == [1, 3, 2]
assert invert_ans([2, 3, 1]) == [3, 1, 2]
print("Bài 1–3 OK")

Bài 1–3 OK


In [27]:
# Bài 4
def my_k_empty_ans(bulbs, k):
    n, size = len(bulbs), k + 1
    nb = n // size + 1
    lo, hi = [float('inf')] * nb, [float('-inf')] * nb
    for day, x in enumerate(bulbs, 1):
        p = x - 1
        b = p // size
        if p < lo[b]:
            lo[b] = p
            if b > 0 and hi[b - 1] == p - size:
                return day
        if p > hi[b]:
            hi[b] = p
            if b + 1 < nb and lo[b + 1] == p + size:
                return day
    return -1

# Bài 5
def last_day_with_pair_ans(bulbs, k):
    n = len(bulbs)
    days = invert_ans(bulbs)
    best = -1
    for left in range(n - k - 1):
        right = left + k + 1
        start = max(days[left], days[right])
        inner = days[left + 1:right]
        end = min(inner) - 1 if inner else n
        if start <= end:
            best = max(best, end)
    return best

def last_day_brute(bulbs, k):
    n = len(bulbs)
    state = [False] * (n + 1)
    best = -1
    for day, x in enumerate(bulbs, 1):
        state[x] = True
        prev = None
        for p in range(1, n + 1):
            if state[p]:
                if prev is not None and p - prev - 1 == k:
                    best = day
                prev = p
    return best

random.seed(719)
for _ in range(800):
    n = random.randint(1, 10)
    bb = random.sample(range(1, n + 1), n)
    for k in range(n + 1):
        assert my_k_empty_ans(bb, k) == k_empty_brute(bb, k)
        assert last_day_with_pair_ans(bb, k) == last_day_brute(bb, k)
print("Bài 4–5 OK (đối chiếu brute force)")

Bài 4–5 OK (đối chiếu brute force)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "hàng xóm có thứ tự" (bisect):**

```python
on = []
for day, x in enumerate(bulbs, 1):
    i = bisect_left(on, x)
    if i > 0 and x - on[i-1] - 1 == k: return day
    if i < len(on) and on[i] - x - 1 == k: return day
    on.insert(i, x)
return -1
```

**Template "chia xô kích thước D = k + 1":**

```python
b = p // D
if p < lo[b]: lo[b] = p; kiểm tra hi[b-1] == p - D
if p > hi[b]: hi[b] = p; kiểm tra lo[b+1] == p + D
```

**Template "đảo mảng + cửa sổ":** `days[x-1] = day`; cửa sổ `(left, left+k+1)` hợp lệ khi mọi ô trong có ngày lớn hơn cả hai đầu; nhảy `left` tới ô hỏng.

**Checklist trước khi submit:**

- [ ] Đọc đúng chiều dữ liệu: `bulbs[i]` = **vị trí** bật vào **ngày `i+1`**.
- [ ] "Đúng `k` bóng ở giữa" ⇔ hai vị trí cách nhau `k + 1`.
- [ ] Kiểm tra cả hàng xóm trái **và** phải.
- [ ] Xử lý `k + 1 >= n` → `-1`; `k = 0` → hai bóng kề nhau.
- [ ] Ngày đếm từ 1 (`enumerate(..., 1)`).
- [ ] Cửa sổ trượt: lấy **min** trên mọi cửa sổ hợp lệ.
- [ ] Đối chiếu với brute force trên hoán vị ngẫu nhiên.
- [ ] Tên method đúng: `kEmptySlots(self, bulbs, k)` (bài Premium — đối chiếu lại nếu có tài khoản).